# **Стиринг активациями: практика**

В уроке «Стиринг активациями и representation engineering» вектор стиринга строился как разность средних активаций на двух контрастных наборах, $v = \bar h_{+} - \bar h_{-}$, а вмешательство записывалось как $h \leftarrow h + \alpha v$ на выбранном слое $\ell$. В toy-примере урока все считалось в пространстве размерности 4. Здесь тот же рецепт применяется к GPT-2 small (12 блоков, residual stream размерности 768), и у каждой из трех настроек урока — направления $v$, силы $\alpha$ и слоя $\ell$ — появляется измеримый эффект.

В этой практике вы:

* построите вектор стиринга «позитивный тон минус негативный тон» как разность средних активаций, как в ActAdd и CAA;
* прибавите его к residual stream на выбранном слое и измерите сдвиг логитов позитивных слов против негативных;
* сравните вектор стиринга со случайными направлениями той же нормы;
* переберете слой $\ell$ и силу $\alpha$ и посмотрите, сколько стоит вмешательство: насколько растет loss модели на нейтральном тексте;
* посмотрите на генерации и увидите, где текст перестает быть связным.

Статьи, на которых держится практика: [Turner et al., «Steering Language Models With Activation Engineering» (ActAdd)](https://arxiv.org/abs/2308.10248), [Panickssery et al., «Steering Llama 2 via Contrastive Activation Addition» (CAA)](https://arxiv.org/abs/2312.06681), [Zou et al., «Representation Engineering: A Top-Down Approach to AI Transparency»](https://arxiv.org/abs/2310.01405). Обзор методов локализации и стиринга с размеченным списком работ: [Zhang et al., «Locate, Steer, and Improve: A Practical Survey of Actionable Mechanistic Interpretability in Large Language Models»](https://arxiv.org/abs/2601.14004) ([список работ на GitHub](https://github.com/rattlesnakey/Awesome-Actionable-MI-Survey)).

Числовые ответы заданий вводятся в уроке «Стиринг активациями: Практика». Тетрадь работает и на CPU, на GPU в Colab она проходит быстрее.

# **Часть 1. Модель и данные**

In [ ]:
%pip install transformers==5.18.0 -q

In [ ]:
import re
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import GPT2TokenizerFast, GPT2LMHeadModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2').to(device).eval()
n_layers = model.config.n_layer
d_model = model.config.n_embd
print(device, n_layers, d_model)

Контрастные наборы — восемь пар фраз с одинаковым строением: в фразе класса «плюс» позитивный тон, в парной фразе класса «минус» негативный. Все фразы кончаются на « and», поэтому активация берется на одном и том же токене, и разность средних не несет различия в последнем слове. Нейтральные промпты — на них измеряется эффект стиринга. Целевые токены — шесть позитивных и шесть негативных прилагательных, каждое из одного токена GPT-2.

In [ ]:
plus_prompts = [
    "I love this. It is wonderful and",
    "What a great day, I feel happy and",
    "This film was excellent, truly beautiful and",
    "She is kind, friendly and",
    "The food was delicious, amazing and",
    "I adore my friends, they are lovely and",
    "The concert was fantastic, joyful and",
    "Our trip was perfect, pleasant and",
]
minus_prompts = [
    "I hate this. It is awful and",
    "What a terrible day, I feel sad and",
    "This film was horrible, truly ugly and",
    "She is cruel, rude and",
    "The food was disgusting, nasty and",
    "I despise my enemies, they are hateful and",
    "The concert was dreadful, miserable and",
    "Our trip was a disaster, unpleasant and",
]
neutral_prompts = [
    "I went to the store and",
    "The weather today is",
    "My neighbor told me that the movie was",
    "Yesterday I read a book, and it was",
    "The meeting this morning was",
    "When I opened the door, I felt",
    "The new restaurant downtown is",
    "After the game, everyone said it was",
]
reference_texts = [
    "The city council met on Tuesday to discuss the new budget for the public library.",
    "He walked to the station, bought a ticket and waited for the train to arrive.",
    "The report describes the results of the survey conducted in three regions last year.",
    "Water boils at one hundred degrees Celsius at sea level.",
    "The students opened their books and started reading the second chapter.",
    "She parked the car near the bank and went inside to pay the bill.",
]
positive_words = [" great", " good", " wonderful", " happy", " amazing", " beautiful"]
negative_words = [" terrible", " bad", " awful", " sad", " horrible", " ugly"]
pos_ids = torch.tensor([tokenizer.encode(w)[0] for w in positive_words], device=device)
neg_ids = torch.tensor([tokenizer.encode(w)[0] for w in negative_words], device=device)
print([len(tokenizer.encode(w)) for w in positive_words + negative_words])

# **Часть 2. Вектор стиринга как разность средних**

Активацию слоя $\ell$ берем на входе в блок $\ell$: это `hidden_states[ℓ]` из выхода модели с `output_hidden_states=True`, содержимое residual stream перед блоком (`hidden_states[0]` — сумма эмбеддингов токена и позиции). Берется последняя позиция промпта. Функция `steering_vector(layer)` возвращает $v = \bar h_{+} - \bar h_{-}$, формулу из урока.

In [ ]:
@torch.no_grad()
def last_token_resid(prompts, layer):
    rows = []
    for p in prompts:
        ids = tokenizer(p, return_tensors='pt').input_ids.to(device)
        rows.append(model(ids, output_hidden_states=True).hidden_states[layer][0, -1])
    return torch.stack(rows)


def steering_vector(layer):
    return last_token_resid(plus_prompts, layer).mean(0) - last_token_resid(minus_prompts, layer).mean(0)

**Задание 1. Постройте вектор стиринга $v$ на слое $\ell = 6$. Чему равна его норма $\lVert v\rVert$? Ответ округлите до одного знака после точки.**

In [ ]:
# Ваш код здесь

Проверим утверждение из toy-примера урока: прибавление ровно одного $v$ переносит центроид «минуса» в центроид «плюса», поэтому проекция на $\hat v = v/\lVert v\rVert$ растет ровно на $\lVert v\rVert$.

In [ ]:
h_plus = last_token_resid(plus_prompts, 6).mean(0)
h_minus = last_token_resid(minus_prompts, 6).mean(0)
v_hat = v6 / v6.norm()
print(round((h_minus @ v_hat).item(), 3), round(((h_minus + v6) @ v_hat).item(), 3), round((h_plus @ v_hat).item(), 3))

Норма вектора стиринга зависит от слоя: норма residual stream растет к поздним блокам, и разность центроидов растет вместе с ней. Поэтому одна и та же сила $\alpha$ на разных слоях означает сдвиг разной длины, а $\alpha = 1$ на любом слое означает одно и то же — перенос центроида «минуса» в центроид «плюса».

In [ ]:
vectors = {layer: steering_vector(layer) for layer in range(n_layers)}
print([round(vectors[layer].norm().item(), 2) for layer in range(n_layers)])

# **Часть 3. Вмешательство и его эффект**

Вмешательство $h \leftarrow h + \alpha v$ делается forward pre-hook на блоке $\ell$: к входу блока во всех позициях прибавляется $\alpha v$. Эффект измеряется **разностью логитов**: средний логит шести позитивных слов минус средний логит шести негативных на следующем токене, усредненный по восьми нейтральным промптам. Сдвиг разности логитов — эта величина со стирингом минус она же без стиринга.

In [ ]:
def add_steering(layer, vector, alpha):
    def hook(module, args):
        return (args[0] + alpha * vector,) + args[1:]
    return model.transformer.h[layer].register_forward_pre_hook(hook)


@torch.no_grad()
def logit_diff(layer, vector, alpha):
    handle = add_steering(layer, vector, alpha)
    diffs = []
    for p in neutral_prompts:
        ids = tokenizer(p, return_tensors='pt').input_ids.to(device)
        logits = model(ids).logits[0, -1]
        diffs.append(logits[pos_ids].mean() - logits[neg_ids].mean())
    handle.remove()
    return torch.stack(diffs).mean().item()


zero = torch.zeros(d_model, device=device)

**Задание 2. Чему равна разность логитов без стиринга ($\alpha = 0$)? Ответ округлите до трех знаков после точки.**

In [ ]:
# Ваш код здесь

**Задание 3. Добавьте вектор стиринга слоя 6 с силой $\alpha = 1$ на слое 6. На сколько сдвинулась разность логитов относительно прогона без стиринга? Ответ округлите до трех знаков после точки.**

In [ ]:
# Ваш код здесь

**Задание 4. Повторите то же с силой $\alpha = -1$. Чему равен сдвиг разности логитов? Ответ со знаком, округлите до трех знаков после точки.**

In [ ]:
# Ваш код здесь

Знак $\alpha$ меняет знак сдвига, как и обещает урок: отрицательная сила гасит свойство. Модули сдвигов при $\alpha = 1$ и $\alpha = -1$ при этом не совпадают: модель нелинейна, и после слоя 6 идут еще шесть блоков, поэтому линейность из toy-примера здесь выполняется только для проекции на слое вмешательства, а не для логитов.

# **Часть 4. Контроль: случайные направления той же нормы**

Сдвиг из задания 3 может объясняться самим фактом вмешательства: любой вектор с нормой $\lVert v\rVert$ меняет активации и логиты. Контроль — 20 случайных направлений (сид 0), отнормированных до той же нормы, что у $v$ слоя 6, на том же слое и с той же силой $\alpha = 1$.

**Задание 5. Чему равен наибольший по модулю сдвиг разности логитов среди 20 случайных направлений? Ответ — модуль сдвига, округлите до трех знаков после точки.**

In [ ]:
# Ваш код здесь

# **Часть 5. Перебор слоя и силы**

Теперь переберем слой $\ell$ от 0 до 11 и силу $\alpha$ из сетки $\{-4, -2, -1, 0.5, 1, 2, 4\}$; на каждом слое берется свой вектор стиринга. Вторая величина — **рост loss**: средняя кросс-энтропия следующего токена на шести нейтральных предложениях со стирингом минус она же без стиринга, в натах. Она показывает, насколько вмешательство уводит модель от ее обычного распределения на тексте, не связанном с тоном.

In [ ]:
@torch.no_grad()
def lm_loss(layer, vector, alpha):
    handle = add_steering(layer, vector, alpha)
    losses = []
    for s in reference_texts:
        ids = tokenizer(s, return_tensors='pt').input_ids.to(device)
        losses.append(model(ids, labels=ids).loss)
    handle.remove()
    return torch.stack(losses).mean().item()


alphas = [-4, -2, -1, 0.5, 1, 2, 4]
base_loss = lm_loss(0, zero, 0.0)
shift_table = np.array([[logit_diff(layer, vectors[layer], a) - base_diff for a in alphas] for layer in range(n_layers)])
loss_table = np.array([[lm_loss(layer, vectors[layer], a) - base_loss for a in alphas] for layer in range(n_layers)])
print(round(base_loss, 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, table, title, cmap in [
    (axes[0], shift_table, 'Logit-difference shift (positive minus negative words)', 'RdBu_r'),
    (axes[1], loss_table, 'Loss increase on neutral text, nats', 'Reds'),
]:
    lim = np.abs(table).max()
    im = ax.imshow(table, cmap=cmap, aspect='auto', vmin=-lim if cmap == 'RdBu_r' else 0, vmax=lim)
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            ax.text(j, i, f'{table[i, j]:.2f}', ha='center', va='center', fontsize=7)
    ax.set_xticks(range(len(alphas)), [str(a) for a in alphas])
    ax.set_yticks(range(n_layers))
    ax.set_xlabel('steering strength alpha')
    ax.set_ylabel('layer')
    ax.set_title(title, fontsize=10)
    fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

**Задание 6. На каком слое (номер от 0 до 11) сдвиг разности логитов при $\alpha = 1$ наибольший?**

In [ ]:
# Ваш код здесь

**Задание 7. Насколько вырос loss на нейтральном тексте при стиринге на слое 11 с силой $\alpha = 4$? Ответ в натах, округлите до трех знаков после точки.**

In [ ]:
# Ваш код здесь

Сдвиг на слое 0 близок к нулю: на входе в первый блок активация последнего токена « and» еще не видела контекста, и векторы «плюса» и «минуса» почти совпадают. Чем позже слой, тем больше норма $v$ и тем больше сдвиг разности логитов при той же $\alpha$, но тем больше и рост loss. На поздних слоях residual stream уже близок к распределению следующего токена, и вектор стиринга там работает почти как прямая добавка к логитам нужных слов. В генерациях ниже видно, к чему это приводит при большой силе.

# **Часть 6. Генерации**

Для нескольких пар (слой, сила) модель продолжает каждый нейтральный промпт на 30 токенов жадным декодированием со стирингом во всех позициях. Две величины по восьми продолжениям: число продолжений, где встретилось хотя бы одно слово из позитивного словаря, и доля уникальных слов в продолжении (мера повторов: при зацикливании она падает).

In [ ]:
positive_lexicon = {'great', 'good', 'wonderful', 'happy', 'amazing', 'beautiful', 'love', 'lovely', 'best',
                    'nice', 'excellent', 'fun', 'perfect', 'fantastic', 'enjoy', 'enjoyed', 'glad', 'joy',
                    'delightful', 'pleasant', 'awesome', 'loved', 'kind'}


@torch.no_grad()
def generate(layer, vector, alpha, n_tokens=30):
    handle = add_steering(layer, vector, alpha)
    texts = []
    for p in neutral_prompts:
        ids = tokenizer(p, return_tensors='pt').input_ids.to(device)
        out = model.generate(ids, max_new_tokens=n_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        texts.append(tokenizer.decode(out[0, ids.shape[1]:]))
    handle.remove()
    return texts


def describe(texts):
    words = [re.findall(r'[a-z]+', t.lower()) for t in texts]
    n_positive = sum(any(w in positive_lexicon for w in ws) for ws in words)
    unique_share = float(np.mean([len(set(ws)) / max(len(ws), 1) for ws in words]))
    return n_positive, round(unique_share, 2)


for layer, alpha in [(0, 0.0), (6, 1.0), (6, 4.0), (11, 1.0), (11, 4.0)]:
    texts = generate(layer, vectors[layer], alpha)
    print(f'layer {layer}, alpha {alpha}:', describe(texts))
    for t in texts[1:4]:
        print('   ', repr(t[:110]))

Сравните строки вывода. Без стиринга (слой 0, $\alpha = 0$) позитивные слова встречаются в части продолжений сами по себе. На слое 11 с $\alpha = 4$ доля уникальных слов падает, и текст сводится к повтору одного слова: это и есть уход активаций за пределы распределения, о котором предупреждает урок. Рабочий коридор силы $\alpha$ — там, где сдвиг разности логитов уже есть, а рост loss и доля повторов еще малы; по таблицам части 5 его подбирают для каждого слоя отдельно.

Спасибо за работу!